## AutoPytorch

In [ ]:
# =========================
# INSTALAR AUTO-PYTORCH
# =========================
#%pip install autoPyTorch==0.2.1 scikit-learn==0.24.2

# =========================
# IMPORTS
# =========================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

from autoPyTorch.api.tabular_classification import TabularClassificationTask

# =========================
# CARREGAR DATASET
# =========================
df = pd.read_csv("data/creditcard.csv")

print("Dataset carregado:")
print(df.head())

# =========================
# LIMPEZA
# =========================
df = df.dropna(subset=['Class'])

X = df.drop(['Class', 'Time'], axis=1).to_numpy()
y = df['Class'].astype(int).to_numpy()

# =========================
# DIVISÃO
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# AUTO-PYTORCH
# =========================
automl = TabularClassificationTask()

automl.search(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    optimize_metric='accuracy',
    total_walltime_limit=300,   # tempo total (5 min)
    func_eval_time_limit_secs=50
)

# =========================
# RESULTADOS
# =========================
y_pred = automl.predict(X_test)

print("\n===== MÉTRICAS =====")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_test, y_pred, pos_label=1, zero_division=0), 4))

## AutoSKlearn

In [ ]:
# =========================
# INSTALAR AUTO-SKLEARN
# =========================
#%pip install autosklearn

# =========================
# IMPORTS
# =========================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

import autosklearn.classification

# =========================
# CARREGAR DATASET
# =========================
df = pd.read_csv("data/creditcard.csv")

print("Dataset carregado:")
print(df.head())

# =========================
# LIMPEZA
# =========================
df = df.dropna(subset=['Class'])

X = df.drop(['Class', 'Time'], axis=1)
y = df['Class'].astype(int)

# =========================
# DIVISÃO
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# AUTO-SKLEARN
# =========================
automl = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=300,   # tempo total (5 min)
    per_run_time_limit=60,         # tempo por modelo
    memory_limit=4096,             # memória (MB)
    seed=42
)

automl.fit(X_train, y_train)

# =========================
# RESULTADOS
# =========================
y_pred = automl.predict(X_test)

print("\n===== MÉTRICAS =====")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, pos_label=1, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_test, y_pred, pos_label=1, zero_division=0), 4))

# =========================
# MODELOS USADOS
# =========================
print("\n===== MODELOS =====")
print(automl.show_models())

## LazyPredict

In [ ]:
# =========================
# INSTALAR LAZYPREDICT
# =========================
#%pip install lazypredict

# =========================
# IMPORTS
# =========================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from lazypredict.Supervised import LazyClassifier

# =========================
# CARREGAR DATASET
# =========================
df = pd.read_csv("data/creditcard.csv")

print("Dataset carregado:")
print(df.head())

# =========================
# LIMPEZA
# =========================
df = df.dropna(subset=['Class'])

X = df.drop(['Class', 'Time'], axis=1)
y = df['Class'].astype(int)

# =========================
# TF-IDF
# =========================
#vectorizer = TfidfVectorizer(
#    max_features=3000,   # um pouco menor pra rodar rápido
#    ngram_range=(1,2)
#)

#X = vectorizer.fit_transform(X).toarray()

# =========================
# DIVISÃO
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# LAZYPREDICT
# =========================
clf = LazyClassifier(verbose=0, ignore_warnings=True)

models, predictions = clf.fit(X_train, X_test, y_train, y_test)

# =========================
# RESULTADOS
# =========================
print("\n===== RESULTADOS =====")
print(models)

print("\n===== MÉTRICAS =====")
all_metrics = []

for model_name, model in clf.models.items():
    y_pred = model.predict(X_test)

    all_metrics.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
    })

metrics_df = (
    pd.DataFrame(all_metrics)
    .sort_values("F1 Score", ascending=False)
    .reset_index(drop=True)
)

print(metrics_df)